# High-performance computing colocation — parameter sweep (design space exploration)

This notebook demonstrates **design-space exploration** on a notional **high-performance computing colocation** electrical load model: we sweep **equipment electrical load** (servers, storage, and networking) and **auxiliary cooling** power while holding **grid import** and **cooling envelope** limits fixed, then summarize pass/fail outcomes.

ThunderGraph pieces in play:

- **`HpcDatacenterProgram.instantiate()`** → **`ConfiguredModel`**
- **`ConfiguredModel.evaluate(inputs={ ValueSlot: Quantity, … })`** — the recommended façade (lazy compile on first use)
- **Composable Level-1** **`Requirement`** tree: **`model.requirement_package`** nests `L1HpcRoot` → `L1HpcRequirements`; leaf **`model.requirement`** with **`requirement_input`**, **`requirement_attribute`** (margins), and **`requirement_accept_expr`**, wired with **`allocate`**

**Sweep performance:** static graph validation is useful once; for repeated points use **`evaluate(..., validate=False)`** after you have validated the graph once (same contract as the library docstring).

The model lives in `examples/hpc_datacenter/` so this notebook can stay short: imports, helpers, sweep, results.

## Path setup

The `thundergraph-model` package and `examples/` must be importable. If you open this notebook from another working directory, this cell walks upward until it finds `tg_model/`.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path


def thundergraph_package_root(start: Path) -> Path:
    p = start.resolve()
    for _ in range(24):
        if (p / "tg_model" / "__init__.py").is_file():
            return p
        nested = p / "thundergraph-model"
        if (nested / "tg_model" / "__init__.py").is_file():
            return nested
        if p.parent == p:
            break
        p = p.parent
    return start.resolve()


_cwd = Path.cwd().resolve()
_pkg = thundergraph_package_root(_cwd)
_examples = _pkg / "examples"
for _path in (_pkg, _examples):
    s = str(_path)
    if s in sys.path:
        sys.path.remove(s)
    sys.path.insert(0, s)

## Model + sweep helpers

All scenario state is passed as **arguments**. Each sweep iteration reuses one **`ConfiguredModel`** instance so the compiled graph is cached; inputs change per point. **`ValueSlot` handles always come from the same `cm` you evaluate** (do not reuse slots across different `instantiate()` calls).

In [2]:
import itertools
from dataclasses import dataclass

from unitflow import Quantity
from unitflow.catalogs.si import kW

from hpc_datacenter import HpcDatacenterProgram
from tg_model.execution.configured_model import ConfiguredModel
from tg_model.execution.evaluator import RunResult
from tg_model.execution.graph_compiler import compile_graph
from tg_model.execution.validation import validate_graph


def make_evaluation_inputs(
    cm: ConfiguredModel,
    *,
    equipment_load_kw: float,
    auxiliary_cooling_kw: float,
    grid_cap_kw: float,
    max_cooling_kw: float,
) -> dict[object, Quantity]:
    """Return one evaluation input map keyed by **this** `cm`'s handles."""
    fac = cm.facility
    return {
        cm.equipment_electrical_load_kw: Quantity(equipment_load_kw, kW),
        cm.auxiliary_cooling_load_kw: Quantity(auxiliary_cooling_kw, kW),
        fac.grid_import_capacity_kw: Quantity(grid_cap_kw, kW),
        fac.max_cooling_kw: Quantity(max_cooling_kw, kW),
    }


@dataclass(frozen=True)
class SweepRow:
    equipment_load_kw: float
    auxiliary_cooling_kw: float
    total_kw: float
    passed: bool


def run_facility_power_sweep(
    *,
    equipment_load_kw_values: tuple[float, ...],
    auxiliary_cooling_kw_values: tuple[float, ...],
    grid_cap_kw: float,
    max_cooling_kw: float,
) -> list[SweepRow]:
    """Run a full factorial sweep; validate the graph once, then evaluate with ``validate=False``."""
    cm = HpcDatacenterProgram.instantiate()
    graph, _handlers = compile_graph(cm)
    first = validate_graph(graph, configured_model=cm)
    if not first.passed:
        raise RuntimeError(
            "Graph validation failed before sweep: "
            + "; ".join(f"{f.category}: {f.message}" for f in first.failures)
        )

    rows: list[SweepRow] = []
    for eq_kw, cool_kw in itertools.product(equipment_load_kw_values, auxiliary_cooling_kw_values):
        inputs = make_evaluation_inputs(
            cm,
            equipment_load_kw=eq_kw,
            auxiliary_cooling_kw=cool_kw,
            grid_cap_kw=grid_cap_kw,
            max_cooling_kw=max_cooling_kw,
        )
        result: RunResult = cm.evaluate(inputs=inputs, validate=False)
        total_out = result.outputs[cm.total_facility_kw.stable_id]
        rows.append(
            SweepRow(
                equipment_load_kw=eq_kw,
                auxiliary_cooling_kw=cool_kw,
                total_kw=float(total_out.magnitude),
                passed=result.passed,
            )
        )
    return rows

## Run the sweep

The grid cap is **50 kW** import and auxiliary cooling is capped at **12 kW** in this toy envelope. We sweep equipment electrical load **10–40 kW** and auxiliary cooling **2–14 kW** in **5 kW** steps (coarse grid for a readable table).

In [3]:
rows = run_facility_power_sweep(
    equipment_load_kw_values=(10.0, 15.0, 20.0, 25.0, 30.0, 35.0, 40.0),
    auxiliary_cooling_kw_values=(2.0, 7.0, 12.0, 14.0),
    grid_cap_kw=50.0,
    max_cooling_kw=12.0,
)

print(f"{'Equipment kW':>14} {'Cooling kW':>12} {'Total kW':>10} {'PASS':>5}")
print("-" * 48)
for row in rows:
    print(
        f"{row.equipment_load_kw:14.1f} {row.auxiliary_cooling_kw:12.1f} {row.total_kw:10.1f} {str(row.passed):>5}"
    )

pass_count = sum(1 for r in rows if r.passed)
print(f"\nPassing combinations: {pass_count} / {len(rows)}")

  Equipment kW   Cooling kW   Total kW  PASS
------------------------------------------------
          10.0          2.0       12.0  True
          10.0          7.0       17.0  True
          10.0         12.0       22.0  True
          10.0         14.0       24.0 False
          15.0          2.0       17.0  True
          15.0          7.0       22.0  True
          15.0         12.0       27.0  True
          15.0         14.0       29.0 False
          20.0          2.0       22.0  True
          20.0          7.0       27.0  True
          20.0         12.0       32.0  True
          20.0         14.0       34.0 False
          25.0          2.0       27.0  True
          25.0          7.0       32.0  True
          25.0         12.0       37.0  True
          25.0         14.0       39.0 False
          30.0          2.0       32.0  True
          30.0          7.0       37.0  True
          30.0         12.0       42.0  True
          30.0         14.0       44.0 False
      